<a href="https://colab.research.google.com/github/COL-O/POTOK/blob/arena%2F3ce48adc-potok/notebooks/potok-colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# POTOK Colab: вне-Actions иммунная проверка

**Зачем.** CI «POTOK Immune Check» — главный голос дома, но он один, на машинах GitHub. Эта тетрадь — **вторая среда**: Google Colab. Именно в Colab основатель увидел 5–7 красных теста, которые CI проглатывал (без `pipefail` у `tee`) — см. `docs/PROJECT-STATUS.md`, задача №6.

**Что делает по порядку:** клон `main` → установка Go 1.22.12 → `go test -race ./...` → 5-кратная прогн по бывшему флейку фронта → `go build` → запуск `examples/simple`, `examples/drone` и бинарь организма (4 такта).

**Чего не делает:** не пишет в репозиторий, не кладёт секреты, не трогает Actions. Всё, что видите красным, пишите в задачу №6 или в общий чат №12 (через мост, не из ноутбука).

> КОН 2: вывод — дословный. Факт и суждение врозь: ячейки печатают факт, вывод делаете вы.


In [ ]:
# Клон дома (свежий main)
import os, subprocess
subprocess.run(['rm', '-rf', '/content/POTOK'], check=False)
subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'main',
                'https://github.com/COL-O/POTOK', '/content/POTOK'], check=True)
os.chdir('/content/POTOK')
subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], check=True)


In [ ]:
# Go 1.22.12 из официального архива (apt даёт старый)
import subprocess, os
subprocess.run('curl -sL https://dl.google.com/go/go1.22.12.linux-amd64.tar.gz | tar -C /usr/local -xz', shell=True, check=True)
os.environ['PATH'] = '/usr/local/go/bin:' + os.environ['PATH']
print(subprocess.run(['/usr/local/go/bin/go', 'version'], capture_output=True, text=True).stdout)


In [ ]:
# Иммунитет №1: полный прогон с детектором гонок (как шаг CI)
import subprocess, os
os.chdir('/content/POTOK')
os.environ['PATH'] = '/usr/local/go/bin:' + os.environ['PATH']
r = subprocess.run(['go', 'test', '-race', '-count=1', './...'], capture_output=True, text=True)
print(r.stdout[-8000:])
print(r.stderr[-2000:])
print('VERDICT:', 'PASS' if r.returncode == 0 else f'FAIL (код {r.returncode})')


In [ ]:
# Иммунитет №2: бывший флейк фронта — 5 подряд
import subprocess, os
os.chdir('/content/POTOK')
os.environ['PATH'] = '/usr/local/go/bin:' + os.environ['PATH']
for i in range(1, 6):
    r = subprocess.run(['go', 'test', '-race', '-count=1',
                        '-run', 'TestRouterRoutineVetoAndQuarantine',
                        './modules/front/'], capture_output=True, text=True)
    print(f'прогон {i}:', 'PASS' if r.returncode == 0 else 'FAIL')


In [ ]:
# Иммунитет №3: сборка бинаря и два примера + 4 такта организма
import subprocess, os
os.environ['PATH'] = '/usr/local/go/bin:' + os.environ['PATH']
os.chdir('/content/POTOK')
subprocess.run(['go', 'build', '-o', 'bin/organism', './cmd/organism/...'], check=True)
print('--- examples/simple'); subprocess.run(['go', 'run', './examples/simple/main.go'], check=True)
print('--- examples/drone');  subprocess.run(['go', 'run', './examples/drone/main.go'],  check=True)
print('--- 4 такта организма'); subprocess.run(['./bin/organism'], check=True)


In [ ]:
# Сводка вывода для дома: /content/potok-colab-report.md
# (файл никуда не отправляется сам — его читаете вы)
import subprocess, os, datetime
os.chdir('/content/POTOK')
ver_go = subprocess.run(['/usr/local/go/bin/go', 'version'], capture_output=True, text=True).stdout.strip()
sha = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip()
report = f"""# POTOK Colab immune lane — сводка
- Дата: {datetime.date.today().isoformat()}
- main @ {sha}, {ver_go}
- Полный прогон: см. ячейку "go test -race ./..." выше (PASS/FAIL напечатан там)
- Флейк фронта: 5 прогонов выше
- Сборка и 4 такта: вывод выше
\"\"\"
with open('/content/potok-colab-report.md', 'w', encoding='utf-8') as f:
    f.write(report)
print(report)


## Куда сообщать

Красное — в задачу №6 («Колаб-прогон основателя…») или репликой в чат №12 через мост дома (`inbox/<имя>.md` + push). Если факт коммитабельный (например, новая серия красных) — задача №6 с дословным выводом, как было в день памяти (см. `docs/PROJECT-STATUS.md`).

## Необязательно: TPU v5e

В Colab бесплатно доступен рантайм CPU, T4 и **TPU v5e-1** (≈3 единицы/час, квота — «несколько коротких прогонов»). Для POTOK сегодняшней формы (Go, mmap) TPU ничего не даст — организм не использует JAX/тензоры. TPU-смысл появится, если круг решит поднять Python/JAX-сторону (например, SINDy с библиотекой многочленов в духе роста, или обучение KAN). Это отдельная задача с словом основателя — не кроим архитектуру в тетради.
